# 💊 Pharmaceutical Supply Chain Optimization
**Objective:** Reduce stockouts, identify overstock, and recommend optimal order quantities.
---

## Phase 3 — Data Cleaning & Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
ACCENT = '#00d4aa'; RED='#ff4757'; YELLOW='#ffd166'; BLUE='#5e9eff'
PALETTE = [ACCENT, BLUE, YELLOW, RED, '#6c63ff', '#2ed573', '#ff9f43']

df = pd.read_csv('../data/inventory_data.csv')
df['DateTime'] = pd.to_datetime(df['DateTime'])
print('Shape:', df.shape)
df.head()

In [ ]:
print('Null values:\n', df.isnull().sum())
print('\nDescribe:')
df.describe()

## Phase 4 — Exploratory Analysis

In [ ]:
# Top 10 Drugs by Demand
top_drugs = df.groupby('Product')['Demand'].sum().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(12,6))
bars = ax.bar(top_drugs.index, top_drugs.values, color=PALETTE*2, edgecolor='none')
ax.set_title('Top 10 Drugs by Total Demand', fontsize=14, fontweight='bold', color='white')
ax.set_facecolor('#111520'); fig.patch.set_facecolor('#0a0d14')
ax.tick_params(axis='x', rotation=30, colors='#aaa'); ax.tick_params(axis='y', colors='#aaa')
plt.tight_layout()
plt.savefig('../visualizations/top_drugs_demand.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Therapy Area
therapy = df.groupby('TherapyArea')['Demand'].sum().sort_values(ascending=False)
fig, (ax1, ax2) = plt.subplots(1,2,figsize=(14,6))
ax1.barh(therapy.index, therapy.values, color=PALETTE, edgecolor='none')
ax1.set_title('Demand by Therapy Area', color='white', fontweight='bold'); ax1.set_facecolor('#111520'); ax1.tick_params(colors='#aaa')
wedges,texts,autos = ax2.pie(therapy.values, labels=therapy.index, autopct='%1.1f%%', colors=PALETTE, wedgeprops={'edgecolor':'#0a0d14','linewidth':2})
for t in texts: t.set_color('#aaa'); t.set_fontsize(9)
for a in autos: a.set_color('white'); a.set_fontsize(8)
ax2.set_title('Therapy Area Share', color='white', fontweight='bold'); ax2.set_facecolor('#111520')
fig.patch.set_facecolor('#0a0d14'); plt.tight_layout()
plt.savefig('../visualizations/therapy_area.png', dpi=150, bbox_inches='tight'); plt.show()

## Phase 5 — Core Analytics (Stockout + Overstock + ABC)

In [ ]:
df['Inventory_Gap'] = df['Inventory'] - df['Demand']
df['Stockout_Risk']  = df['Demand'] > df['Inventory']
df['Overstock']      = df['Inventory'] > df['Demand'] * 1.5

print(f'Stockout risk: {df["Stockout_Risk"].mean()*100:.1f}%')
print(f'Overstock:     {df["Overstock"].mean()*100:.1f}%')

gap = df.groupby('Product')['Inventory_Gap'].mean().sort_values()
fig, ax = plt.subplots(figsize=(12,7))
cols = [RED if g<0 else ACCENT for g in gap.values]
ax.barh(gap.index, gap.values, color=cols, edgecolor='none')
ax.axvline(0, color='white', linewidth=0.7, linestyle='--', alpha=0.5)
ax.set_title('Inventory Gap by Product', fontsize=13, fontweight='bold', color='white')
ax.set_facecolor('#111520'); fig.patch.set_facecolor('#0a0d14'); ax.tick_params(colors='#aaa')
plt.tight_layout(); plt.savefig('../visualizations/inventory_gap.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# ABC Analysis
df['Revenue'] = df['Demand'] * df['CostPerUnit_INR']
prod_rev = df.groupby('Product')['Revenue'].sum().sort_values(ascending=False).reset_index()
prod_rev['Cum_Pct'] = prod_rev['Revenue'].cumsum() / prod_rev['Revenue'].sum() * 100
prod_rev['ABC'] = prod_rev['Cum_Pct'].apply(lambda p: 'A' if p<=70 else ('B' if p<=90 else 'C'))

abc_colors = {'A':ACCENT,'B':YELLOW,'C':'#6c63ff'}
fig, ax = plt.subplots(figsize=(12,5))
ax.bar(prod_rev['Product'], prod_rev['Revenue']/1e6, color=[abc_colors[c] for c in prod_rev['ABC']], edgecolor='none')
ax.set_title('ABC Analysis — Revenue Classification', fontsize=13, fontweight='bold', color='white')
ax.set_ylabel('Revenue (INR M)', color='#6e7a8f'); ax.tick_params(axis='x', rotation=35, colors='#aaa'); ax.tick_params(axis='y', colors='#aaa')
ax.set_facecolor('#111520'); fig.patch.set_facecolor('#0a0d14')
from matplotlib.patches import Patch
leg = [Patch(color=ACCENT,label='A - High'),Patch(color=YELLOW,label='B - Medium'),Patch(color='#6c63ff',label='C - Low')]
ax.legend(handles=leg, facecolor='#161c2c', edgecolor='#333', labelcolor='white')
plt.tight_layout(); plt.savefig('../visualizations/abc_analysis.png', dpi=150, bbox_inches='tight'); plt.show()
print(prod_rev.groupby('ABC').size())

## Phase 6 — EOQ Optimization

In [ ]:
ORDERING_COST = 5000
HOLDING_PCT   = 0.20
df['Holding_Cost'] = df['CostPerUnit_INR'] * HOLDING_PCT
df['EOQ'] = np.sqrt((2 * df['Demand'] * ORDERING_COST) / df['Holding_Cost']).round(0).astype(int)

eoq_summary = df.groupby('Product').agg(
    Avg_EOQ      = ('EOQ','mean'),
    Avg_Cost     = ('CostPerUnit_INR','mean'),
    Stockout_Pct = ('Stockout_Risk', lambda x: x.mean()*100)
).round(2).reset_index().sort_values('Stockout_Pct', ascending=False)

print(eoq_summary.head(12).to_string(index=False))

In [ ]:
top_risk = eoq_summary.head(12)
fig, ax1 = plt.subplots(figsize=(14,6)); ax2 = ax1.twinx()
x = range(len(top_risk))
ax1.bar(x, top_risk['Avg_EOQ'], color=ACCENT+'bb', edgecolor=ACCENT, linewidth=1, label='EOQ (units)')
ax2.plot(x, top_risk['Stockout_Pct'], 'o-', color=RED, linewidth=2, markersize=6, label='Stockout %')
ax1.set_xticks(list(x)); ax1.set_xticklabels(top_risk['Product'], rotation=35, ha='right', color='#aaa')
ax1.set_ylabel('EOQ (units)', color=ACCENT); ax2.set_ylabel('Stockout Risk (%)', color=RED)
ax1.set_title('EOQ vs Stockout Risk', fontsize=13, fontweight='bold', color='white')
ax1.set_facecolor('#111520'); fig.patch.set_facecolor('#0a0d14'); ax1.tick_params(colors='#aaa'); ax2.tick_params(colors=RED)
lines1,labs1 = ax1.get_legend_handles_labels(); lines2,labs2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labs1+labs2, facecolor='#161c2c', edgecolor='#333', labelcolor='white')
plt.tight_layout(); plt.savefig('../visualizations/eoq_recommendations.png', dpi=150, bbox_inches='tight'); plt.show()

## Phase 7 — Regional Analysis

In [ ]:
regional = df.groupby('Region').agg(Demand=('Demand','sum'), Inventory=('Inventory','sum'), Stockouts=('Stockout_Risk','sum'), Avg_Delay=('DeliveryDelay_Days','mean')).round(2).reset_index()
regional['Gap'] = regional['Inventory'] - regional['Demand']

fig, axes = plt.subplots(1,2,figsize=(14,6))
x = np.arange(len(regional)); w = 0.35
axes[0].bar(x-w/2, regional['Demand'], w, label='Demand', color=ACCENT+'bb', edgecolor=ACCENT)
axes[0].bar(x+w/2, regional['Inventory'], w, label='Inventory', color=BLUE+'bb', edgecolor=BLUE)
axes[0].set_xticks(x); axes[0].set_xticklabels(regional['Region'], rotation=20, color='#aaa')
axes[0].set_title('Regional Demand vs Inventory', color='white', fontweight='bold')
axes[0].legend(facecolor='#161c2c',edgecolor='#333',labelcolor='white'); axes[0].set_facecolor('#111520'); axes[0].tick_params(colors='#aaa')
axes[1].bar(regional['Region'], regional['Stockouts'], color=RED+'bb', edgecolor=RED)
axes[1].set_title('Stockouts by Region', color='white', fontweight='bold'); axes[1].set_xticklabels(regional['Region'], rotation=20, color='#aaa'); axes[1].set_facecolor('#111520'); axes[1].tick_params(colors='#aaa')
fig.patch.set_facecolor('#0a0d14'); plt.tight_layout()
plt.savefig('../visualizations/regional_analysis.png', dpi=150, bbox_inches='tight'); plt.show()
print(regional.to_string(index=False))

---
## ✅ Analysis Complete
All visualisations saved to `../visualizations/`. Open `../dashboard/inventory_dashboard.html` for the full interactive dashboard.